# AIC 2026 — Qwen3-VL-Reranker-8B rerank server (Colab A100 40 GB)

Notebook này dựng **remote rerank service** cho backend AIC26, đúng kiến trúc đã chốt
trong `reranker.md`: backend Docker (`python:3.11-slim`, không torch/CUDA) chỉ gọi HTTP,
GPU model chạy tách rời như `PE server` hiện tại.

```text
AIC backend (FastAPI, CPU)
│
├── HTTP → PE-Core server            (đã có)
│
└── HTTP → Qwen3-VL-Reranker server  (notebook này)
```

## Vị trí trong pipeline

Reranker chấm điểm cặp `(query, keyframe image)` nên phải chạy ở **frame level**,
sau khi union candidate của PE và **trước** RRF / `group_by_video`:

```text
PE-Core → Milvus top 200 (candidate_k)
                ↓
        Qwen3-VL-Reranker-8B      ← notebook này
                ↓
     visual rank (top 100 = output_k)
                ↓
   RRF với OCR / speech / audio → group_by_video
```

Ba ràng buộc bắt buộc khi ghép vào `search_service.py`:

1. **Over-retrieve**: `candidate_k > output_k` (200 → 100). Rerank đúng 100 frame mà PE
   đã trả về chỉ đổi được thứ tự trong 100 đó.
2. **Fail-open**: reranker timeout/lỗi ⇒ giữ nguyên thứ tự PE, **không** để `image_pe`
   thành list rỗng.
3. **Rerank một lần** bằng query chính, không rerank từng query-expansion variant.

## Chuẩn bị trước khi chạy

- Runtime → Change runtime type → **A100 GPU** (40 GB) + **High-RAM**.
  Checkpoint BF16 ~16,3 GiB nạp qua CPU nên cần ≥ 30 GB RAM hệ thống.
- Colab secret `AIC26_QWEN_RERANKER_TOKEN` — bearer token backend dùng (bắt buộc khi chạy thật).
- Colab secret `HF_TOKEN` — tăng độ ổn định khi tải checkpoint.
- Run all từ trên xuống, copy `PUBLIC_URL` ở cell cuối vào `QWEN_RERANKER_URL` trong `backend/.env`.

Ảnh keyframe được đọc trực tiếp từ URL công khai mà `MediaUrlBuilder` sinh ra
(`.../Keyframes/Keyframes_Lxx/Lxx_Vxxx/001.jpg`), nên **không cần encode lại corpus**
và không cần Qwen3-VL-Embedding.


## 0. CONFIG — chỉ sửa cell này


In [ ]:
# =========================
# CONFIG — chỉ sửa cell này
# =========================

CONFIG = {
    # Model + official implementation. Pin cùng git ref với notebook embedding
    # (Qwen3_VL_Embedding_8B_AIC2026_Colab.ipynb) để hai service dùng chung một
    # bản wrapper chính thức đã kiểm chứng.
    "model_id": "Qwen/Qwen3-VL-Reranker-8B",
    "model_revision": "b212dc8c91a8164aef1ea2de9c1a867611e75c04",
    "qwen_git_ref": "393e2978d27852b0d0230d6994f37f9c15bed73c",
    "official_reranker_sha256": "7f55d5598d0e93585dcd9de7cad522756bb9eed7b1e783a66f8d53a1f7f47e6c",
    "expected_checkpoint_total_size": 17_534_247_392,  # bytes trong model.safetensors.index.json

    # Runtime stack đã chạy được với Qwen3-VL trên Colab 2025.10.
    "transformers_version": "5.14.1",
    "qwen_vl_utils_version": "0.0.14",
    "flash_attn_version": "2.8.3.post1",
    "pillow_version": "11.3.0",
    "attention_implementation": "flash_attention_2",  # tự hạ xuống "sdpa" nếu flash-attn không cài được
    "allow_sdpa_fallback": True,
    "model_weight_dtype": "bfloat16",

    # ---- Semantic contract -------------------------------------------------
    # Instruction mặc định mô tả đúng task của AIC26 (rerank keyframe theo mô tả thị giác).
    # Backend có thể override per-request. Hai baseline chính thức để A/B:
    #   "Given a search query, retrieve relevant candidates that answer the query."   (repo default)
    #   "Retrieval relevant image or text with user's query"                          (model card)
    "default_instruction": (
        "Retrieve the video keyframe that best matches the visual content "
        "described in the user's query."
    ),
    "max_length": 10_240,     # official MAX_LENGTH
    "image_patch_size": 16,   # official; IMAGE_FACTOR = patch_size * 2 = 32
    "min_image_tokens": 4,    # official MIN_PIXELS = 4 * 32 * 32
    # Official MAX_PIXELS = 1800 token. Keyframe 1280x720 (~900 token) vốn đã dưới ngưỡng
    # đó, nên hạ budget là cách rẻ nhất để giảm latency. 768 token ≈ 1174x660 cho 16:9.
    # Chạy cell benchmark rồi chọn lại nếu cần độ nét cao hơn.
    "image_token_budget": 768,

    # ---- Throughput --------------------------------------------------------
    # Official process() chạy tuần tự từng cặp (200 candidate = 200 forward).
    # Notebook này batch bằng chính tokenize()/compute_scores() của wrapper.
    "batch_size": 8,
    "autotune_batch_size": True,
    "batch_candidates": [4, 6, 8, 12, 16],
    "max_vram_fraction": 0.92,
    "max_documents": 400,

    # ---- Image I/O ---------------------------------------------------------
    "download_workers": 16,
    "download_timeout_s": 20.0,
    "max_image_bytes": 16 * 1024 * 1024,
    "image_cache_size": 512,     # PIL đã resize; 768 token ≈ 2,3 MB/ảnh
    "result_cache_size": 128,
    "pre_resize_images": True,   # smart-resize trong worker thread thay vì trên luồng chính

    # ---- Server ------------------------------------------------------------
    "port": 8000,
    "service_token_secret": "AIC26_QWEN_RERANKER_TOKEN",
    "warmup_on_start": True,

    # ---- Gates -------------------------------------------------------------
    "run_parity_gate": True,       # batched scores vs official process() từng cặp
    "parity_samples": 6,
    "parity_max_abs_diff": 0.03,
    "run_latency_benchmark": False,
    "benchmark_token_budgets": [256, 512, 768, 1280, 1800],
    "benchmark_documents": 50,
    # Điền URL keyframe thật (MediaUrlBuilder.keyframe_url) để parity/benchmark sát
    # production. Để rỗng thì notebook dùng ảnh synthetic, vẫn chạy được offline.
    "sample_image_urls": [],
    "sample_query": "a man riding a bicycle on a busy street",

    "fail_on_non_a100": True,
    "scratch_root": "/content/aic_qwen3_vl_reranker_8b",
}

IMAGE_FACTOR = CONFIG["image_patch_size"] * 2
CONFIG["min_pixels"] = CONFIG["min_image_tokens"] * IMAGE_FACTOR * IMAGE_FACTOR
CONFIG["max_pixels"] = CONFIG["image_token_budget"] * IMAGE_FACTOR * IMAGE_FACTOR

print({
    "model": CONFIG["model_id"],
    "image_factor": IMAGE_FACTOR,
    "min_pixels": CONFIG["min_pixels"],
    "max_pixels": CONFIG["max_pixels"],
    "instruction": CONFIG["default_instruction"],
})


## 1. Cài dependency, checkout implementation Qwen chính thức, tải cloudflared

Giữ nguyên PyTorch CUDA của Colab. FlashAttention-2 tải wheel prebuilt đúng ABI trước;
không có wheel thì build sm80-only. Nếu vẫn thất bại, notebook **không dừng** mà hạ
xuống `sdpa` (chậm hơn, tốn VRAM hơn, kết quả vẫn đúng).


In [ ]:
from pathlib import Path
import importlib.metadata
import os
import platform
import stat
import subprocess
import sys
import time
import urllib.request


def run_checked(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)


def run_with_heartbeat(cmd, *, env=None, heartbeat_seconds=30, check=True):
    # Stream child output và báo heartbeat nếu build/download tạm thời im lặng.
    command = list(map(str, cmd))
    print("+", " ".join(command), flush=True)
    started = time.monotonic()
    process = subprocess.Popen(command, env=env)
    try:
        while True:
            try:
                return_code = process.wait(timeout=heartbeat_seconds)
                break
            except subprocess.TimeoutExpired:
                elapsed = int(time.monotonic() - started)
                minutes, seconds = divmod(elapsed, 60)
                print(
                    f"[heartbeat] pid={process.pid} vẫn chạy — elapsed={minutes:02d}:{seconds:02d}",
                    flush=True,
                )
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
        raise
    if check and return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code


MINIMAL_PACKAGES = [
    f"transformers=={CONFIG['transformers_version']}",
    "accelerate==1.14.0",
    "huggingface_hub==1.24.0",
    f"qwen-vl-utils=={CONFIG['qwen_vl_utils_version']}",
    f"pillow=={CONFIG['pillow_version']}",
    "safetensors>=0.5,<1",
    "scipy>=1.11",
    "fastapi>=0.115,<1",
    "uvicorn>=0.30,<1",
    "nest_asyncio>=1.6,<2",
    "pydantic>=2.9,<3",
    "requests>=2.32,<3",
    "ninja>=1.11,<2",
    "setuptools>=80.9,<82",
    "wheel>=0.45,<1",
    "packaging>=24,<27",
    "psutil>=6,<8",
]
run_checked([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *MINIMAL_PACKAGES])

# Fail sớm nếu Pillow bị trộn file giữa version cũ/mới trong một kernel đã chạy.
try:
    from PIL import Image as _PIL_IMAGE_CHECK  # noqa: F401
    from PIL import ImageDraw as _PIL_DRAW_CHECK  # noqa: F401
except Exception as exc:
    raise RuntimeError(
        "Pillow runtime không nhất quán. Terminate runtime (không chỉ Restart) rồi Run all lại."
    ) from exc
print("Pillow:", importlib.metadata.version("Pillow"))

QWEN_REPO_DIR = Path("/content/Qwen3-VL-Embedding")
if not (QWEN_REPO_DIR / ".git").exists():
    run_checked([
        "git", "clone", "--filter=blob:none", "--no-tags",
        "https://github.com/QwenLM/Qwen3-VL-Embedding.git",
        str(QWEN_REPO_DIR),
    ])
run_checked([
    "git", "-C", QWEN_REPO_DIR, "remote", "set-url", "origin",
    "https://github.com/QwenLM/Qwen3-VL-Embedding.git",
])
run_checked(["git", "-C", QWEN_REPO_DIR, "fetch", "--depth", "1", "origin", CONFIG["qwen_git_ref"]])
run_checked(["git", "-C", QWEN_REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])

import torch
from packaging.version import Version


def flash_attn_is_ready():
    try:
        if importlib.metadata.version("flash-attn") != CONFIG["flash_attn_version"]:
            return False
        import flash_attn  # noqa: F401
        return True
    except Exception:
        return False


def install_flash_attn():
    flash_version = CONFIG["flash_attn_version"]
    torch_version = Version(torch.__version__.split("+")[0])
    if not torch.version.cuda:
        raise RuntimeError("PyTorch runtime không có CUDA version.")
    torch_cuda_major = int(torch.version.cuda.split(".")[0])
    wheel_cuda_major = 11 if torch_cuda_major == 11 else 12 if torch_cuda_major == 12 else 13
    python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    cxx11_abi = str(torch._C._GLIBCXX_USE_CXX11_ABI).upper()
    platform_tag = f"linux_{platform.uname().machine}"
    wheel_name = (
        f"flash_attn-{flash_version}+cu{wheel_cuda_major}"
        f"torch{torch_version.major}.{torch_version.minor}cxx11abi{cxx11_abi}-"
        f"{python_tag}-{python_tag}-{platform_tag}.whl"
    )
    wheel_url = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/"
        f"v{flash_version}/{wheel_name}"
    )
    wheel_dir = Path("/content/flash_attn_wheels")
    wheel_dir.mkdir(parents=True, exist_ok=True)
    wheel_path = wheel_dir / wheel_name
    partial_path = wheel_path.with_suffix(".whl.partial")
    partial_path.unlink(missing_ok=True)
    print({
        "flash_attn_candidate_wheel": wheel_name,
        "torch": torch.__version__,
        "torch_cuda": torch.version.cuda,
        "cxx11_abi": cxx11_abi,
    })
    download_rc = run_with_heartbeat([
        "curl", "--fail", "--location", "--retry", "2", "--retry-all-errors",
        "--connect-timeout", "15", "--max-time", "300", "--progress-bar",
        "--output", partial_path, wheel_url,
    ], heartbeat_seconds=30, check=False)

    if download_rc == 0 and partial_path.is_file() and partial_path.stat().st_size > 0:
        os.replace(partial_path, wheel_path)
        print("Đã tải prebuilt wheel:", wheel_path.name)
    else:
        partial_path.unlink(missing_ok=True)
        cpu_count = os.cpu_count() or 4
        flash_env = os.environ.copy()
        flash_env.update({
            "PYTHONUNBUFFERED": "1",
            "PIP_DEFAULT_TIMEOUT": "60",
            "FLASH_ATTENTION_FORCE_BUILD": "TRUE",
            "FLASH_ATTN_CUDA_ARCHS": "80",  # A100 only; bỏ sm90/sm100/sm120.
            "MAX_JOBS": str(max(2, min(8, cpu_count // 2))),
            "NVCC_THREADS": "2",
        })
        print("Không có exact prebuilt wheel; build source chỉ cho A100 sm80.", flush=True)
        run_with_heartbeat([
            sys.executable, "-u", "-m", "pip", "wheel", "--verbose",
            "--no-cache-dir", "--no-deps", "--no-build-isolation",
            "--wheel-dir", wheel_dir, f"flash-attn=={flash_version}",
        ], env=flash_env, heartbeat_seconds=30)
        built = sorted(
            wheel_dir.glob(f"flash_attn-{flash_version}*.whl"),
            key=lambda path: path.stat().st_mtime,
        )
        if not built:
            raise RuntimeError("Build xong nhưng không tìm thấy flash-attn wheel.")
        wheel_path = built[-1]
        print("Đã build wheel sm80:", wheel_path)

    run_with_heartbeat([
        sys.executable, "-u", "-m", "pip", "install", "--verbose",
        "--no-deps", "--force-reinstall", wheel_path,
    ], heartbeat_seconds=30)
    if not flash_attn_is_ready():
        raise RuntimeError("flash-attn cài xong nhưng import/version validation thất bại.")


ATTENTION_IMPLEMENTATION = CONFIG["attention_implementation"]
if ATTENTION_IMPLEMENTATION == "flash_attention_2":
    if flash_attn_is_ready():
        print("flash-attn đã sẵn sàng; bỏ qua install.")
    else:
        try:
            install_flash_attn()
        except Exception as exc:
            if not CONFIG["allow_sdpa_fallback"]:
                raise
            print(f"\nWARNING: cài flash-attn thất bại ({exc}); hạ xuống sdpa.\n")
            ATTENTION_IMPLEMENTATION = "sdpa"
print("attention implementation:", ATTENTION_IMPLEMENTATION)

QWEN_REPO_STR = str(QWEN_REPO_DIR.resolve())
if QWEN_REPO_STR not in sys.path:
    sys.path.insert(0, QWEN_REPO_STR)

CLOUDFLARED = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)
print("cloudflared:", CLOUDFLARED, os.path.getsize(CLOUDFLARED), "bytes")


## 2. Secrets và preflight phần cứng

`AIC26_QWEN_RERANKER_TOKEN` là bearer token backend gửi kèm mỗi request. Nếu chưa đặt
secret, notebook sinh token dùng một lần và in ra (chỉ hợp cho thử nghiệm).


In [ ]:
import secrets

import psutil
import torch

try:
    from google.colab import userdata

    def read_secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return os.environ.get(name)
except Exception:
    def read_secret(name):
        return os.environ.get(name)


SERVICE_TOKEN = read_secret(CONFIG["service_token_secret"])
HF_TOKEN = read_secret("HF_TOKEN") or os.environ.get("HF_TOKEN")
if not SERVICE_TOKEN:
    SERVICE_TOKEN = secrets.token_urlsafe(32)
    print(f"WARNING: chưa có secret {CONFIG['service_token_secret']}; token dùng một lần:", SERVICE_TOKEN)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not torch.cuda.is_available():
    raise RuntimeError("Notebook này yêu cầu CUDA GPU (Runtime → Change runtime type → A100).")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
SYSTEM_RAM_GIB = psutil.virtual_memory().total / 2**30
if CONFIG["fail_on_non_a100"] and "A100" not in GPU_NAME.upper():
    raise RuntimeError(f"Yêu cầu A100 nhưng runtime hiện tại là {GPU_NAME}.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError(f"{GPU_NAME} không hỗ trợ BF16.")

CHECKPOINT_GIB = CONFIG["expected_checkpoint_total_size"] / 2**30
if SYSTEM_RAM_GIB < CHECKPOINT_GIB * 1.8:
    print(
        f"WARNING: RAM hệ thống {SYSTEM_RAM_GIB:.1f} GiB có thể không đủ để nạp checkpoint "
        f"{CHECKPOINT_GIB:.1f} GiB qua CPU. Bật High-RAM runtime nếu kernel bị kill."
    )

COMPUTE_DTYPE = torch.bfloat16
DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
SCRATCH_ROOT = Path(CONFIG["scratch_root"])
MODEL_CACHE_ROOT = SCRATCH_ROOT / "hf_cache"
MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "gpu": GPU_NAME,
    "vram_gib": round(GPU_VRAM_GIB, 1),
    "system_ram_gib": round(SYSTEM_RAM_GIB, 1),
    "checkpoint_gib": round(CHECKPOINT_GIB, 2),
    "dtype": str(COMPUTE_DTYPE),
    "hf_token": bool(HF_TOKEN),
})


## 3. Nạp Qwen3-VL-Reranker-8B bằng wrapper chính thức

`Qwen3VLReranker` là **single-tower pointwise reranker**: nó ghép
`<Instruct> / <Query> / <Document>` thành một chat message, lấy `last_hidden_state[:, -1]`
rồi chiếu qua `w_yes - w_no` của `lm_head` và sigmoid ⇒ điểm relevance trong `(0, 1)`.

Vì thế candidate **không cần** đến từ Qwen embedding — PE-Core, CLIP, Elastic đều được.

`processor` được wrapper đặt `padding_side='left'`, đúng thứ mà batching cần: token cuối
của mọi hàng trong batch vẫn là token thật, nên `[:, -1]` không rơi vào padding.


In [ ]:
import gc
import hashlib
import io
import json
import re
import threading
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from huggingface_hub import snapshot_download

MODEL_SNAPSHOT_DIR = Path(snapshot_download(
    repo_id=CONFIG["model_id"],
    revision=CONFIG["model_revision"],
    cache_dir=MODEL_CACHE_ROOT,
    token=HF_TOKEN or None,
    allow_patterns=["*.json", "*.jinja", "*.model", "*.txt", "*.safetensors", "merges.txt", "vocab.json"],
))
MODEL_INDEX_PATH = MODEL_SNAPSHOT_DIR / "model.safetensors.index.json"
if not MODEL_INDEX_PATH.is_file():
    raise FileNotFoundError(f"Thiếu model index: {MODEL_INDEX_PATH}")
MODEL_INDEX = json.loads(MODEL_INDEX_PATH.read_text(encoding="utf-8"))
TOTAL_SIZE = int(MODEL_INDEX.get("metadata", {}).get("total_size", -1))
if TOTAL_SIZE != CONFIG["expected_checkpoint_total_size"]:
    raise RuntimeError(
        f"Checkpoint tensor payload lệch: {TOTAL_SIZE} != {CONFIG['expected_checkpoint_total_size']}. "
        "Revision đã đổi — cập nhật CONFIG['model_revision'] rồi chạy lại."
    )
missing = [name for name in sorted(set(MODEL_INDEX["weight_map"].values()))
           if not (MODEL_SNAPSHOT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Thiếu checkpoint shards: {missing}")


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


OFFICIAL_IMPL_PATH = QWEN_REPO_DIR / "src/models/qwen3_vl_reranker.py"
if not OFFICIAL_IMPL_PATH.is_file():
    raise FileNotFoundError(f"Không thấy official reranker wrapper: {OFFICIAL_IMPL_PATH}")
OFFICIAL_IMPL_SHA256 = sha256_file(OFFICIAL_IMPL_PATH)
if OFFICIAL_IMPL_SHA256 != CONFIG["official_reranker_sha256"]:
    print(
        "WARNING: qwen3_vl_reranker.py khác bản đã kiểm chứng "
        f"({OFFICIAL_IMPL_SHA256} != {CONFIG['official_reranker_sha256']}). "
        "Đọc lại diff trước khi tin parity gate."
    )

from src.models.qwen3_vl_reranker import Qwen3VLReranker  # noqa: E402

reranker_kwargs = dict(
    model_name_or_path=str(MODEL_SNAPSHOT_DIR),
    max_length=CONFIG["max_length"],
    min_pixels=CONFIG["min_pixels"],
    max_pixels=CONFIG["max_pixels"],
    default_instruction=CONFIG["default_instruction"],
    attn_implementation=ATTENTION_IMPLEMENTATION,
    low_cpu_mem_usage=True,
)
try:
    # transformers >= 5 đổi tên torch_dtype -> dtype.
    RERANKER = Qwen3VLReranker(dtype=COMPUTE_DTYPE, **reranker_kwargs)
except TypeError:
    RERANKER = Qwen3VLReranker(torch_dtype=COMPUTE_DTYPE, **reranker_kwargs)

MODEL = RERANKER.model
PROCESSOR = RERANKER.processor
SCORE_LINEAR = RERANKER.score_linear
MODEL.requires_grad_(False).eval()
SCORE_LINEAR.requires_grad_(False).eval()

# Chỉ prefill một lượt rồi đọc last hidden state — KV cache là bộ nhớ thừa.
MODEL.config.use_cache = False
if hasattr(MODEL.config, "text_config"):
    MODEL.config.text_config.use_cache = False

gc.collect()
torch.cuda.empty_cache()

first_parameter = next(MODEL.parameters())
if first_parameter.dtype != COMPUTE_DTYPE or first_parameter.device.type != "cuda":
    raise RuntimeError(f"Model residency/dtype sai: {first_parameter.device}, {first_parameter.dtype}")
loaded_attention = getattr(MODEL.config, "_attn_implementation", None)
if loaded_attention and ATTENTION_IMPLEMENTATION not in str(loaded_attention):
    print(f"WARNING: attention thực tế = {loaded_attention}, yêu cầu = {ATTENTION_IMPLEMENTATION}")

TOKENIZER = PROCESSOR.tokenizer
if getattr(TOKENIZER, "padding_side", None) != "left":
    raise RuntimeError("Processor phải padding_side='left' để last-token scoring đúng khi batch.")

print({
    "model_snapshot": str(MODEL_SNAPSHOT_DIR),
    "official_impl_sha256": OFFICIAL_IMPL_SHA256,
    "attn_implementation": str(loaded_attention),
    "padding_side": TOKENIZER.padding_side,
    "default_instruction": RERANKER.default_instruction,
    "min_pixels": RERANKER.min_pixels,
    "max_pixels": RERANKER.max_pixels,
    "vram_allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 2),
    "vram_reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 2),
})


## 4. Engine: prefetch ảnh song song + batched scoring

Hai thứ khiến bản official không dùng thẳng được cho interactive retrieval:

1. `process()` chạy **một cặp một forward** — 200 candidate = 200 forward.
2. Ảnh được `process_vision_info` tải **tuần tự** ngay trên luồng inference.

Ngoài ra bản `tokenize()` official **hỏng với transformers Qwen3-VL hiện tại**: nó chỉ
cắt và pad `input_ids`, trong khi processor còn trả `mm_token_type_ids` theo từng token và
model bắt buộc mảng này cùng shape với `input_ids`. Hệ quả là `get_rope_index` nhận vào một
Python list chưa cắt, chưa pad rồi chết ngay ở batch=1:

```text
TypeError: only integer tensors of a single element can be converted to an index
```

Engine dưới đây giữ nguyên ngữ nghĩa của wrapper (`format_mm_instruction`,
`truncate_tokens_optimized`, `compute_scores`, cùng min/max pixels) nhưng:

- cắt và pad **mọi mảng song song theo token** đúng cùng vị trí với `input_ids`, rồi vá
  `RERANKER.tokenize` bằng bản đã sửa để `process()` official chạy lại được;

- tải + decode + smart-resize ảnh trong thread pool, cache LRU theo `(url, max_pixels)`;
- đưa **PIL Image** thẳng vào document (wrapper hỗ trợ sẵn) nên không tải lại lần hai;
- gom nhiều cặp vào một forward, tự halve batch khi OOM;
- ảnh hỏng/404 **không** làm hỏng request: doc đó rơi vào `failed[]`, phần còn lại vẫn được chấm.


In [ ]:
import requests
from PIL import Image, ImageDraw, ImageOps
from qwen_vl_utils.vision_process import process_vision_info
from transformers import BatchFeature

try:
    from qwen_vl_utils.vision_process import smart_resize
except Exception:  # phiên bản khác đường dẫn import
    smart_resize = None

MODEL_LOCK = threading.Lock()
IMAGE_CACHE_LOCK = threading.Lock()
RESULT_CACHE_LOCK = threading.Lock()
IMAGE_CACHE = OrderedDict()
RESULT_CACHE = OrderedDict()
STATE = {"batch_size": int(CONFIG["batch_size"])}
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "AIC26-Qwen3VL-Reranker/1.0"})
SESSION.mount("https://", requests.adapters.HTTPAdapter(
    pool_connections=CONFIG["download_workers"], pool_maxsize=CONFIG["download_workers"],
))
SESSION.mount("http://", requests.adapters.HTTPAdapter(
    pool_connections=CONFIG["download_workers"], pool_maxsize=CONFIG["download_workers"],
))
DOWNLOAD_POOL = ThreadPoolExecutor(max_workers=CONFIG["download_workers"])


def fit_image(image):
    # Smart-resize trước, trong worker thread. fetch_image sau đó tính ra đúng
    # kích thước này nên không còn downscale lớn nào chạy trên luồng inference.
    if not CONFIG["pre_resize_images"] or smart_resize is None:
        return image
    try:
        height, width = smart_resize(
            image.height, image.width,
            factor=IMAGE_FACTOR,
            min_pixels=RERANKER.min_pixels,
            max_pixels=RERANKER.max_pixels,
        )
    except Exception:
        return image
    if (height, width) == (image.height, image.width):
        return image
    return image.resize((width, height))


def download_image(url):
    if not isinstance(url, str) or not url.startswith(("http://", "https://")):
        raise ValueError("Chỉ nhận HTTP(S) image URL")
    cache_key = (url, RERANKER.max_pixels)
    with IMAGE_CACHE_LOCK:
        cached = IMAGE_CACHE.get(cache_key)
        if cached is not None:
            IMAGE_CACHE.move_to_end(cache_key)
            return cached
    response = SESSION.get(url, timeout=CONFIG["download_timeout_s"], stream=True)
    response.raise_for_status()
    declared = int(response.headers.get("content-length") or 0)
    if declared > CONFIG["max_image_bytes"]:
        raise ValueError(f"Ảnh quá lớn: {declared} bytes")
    data = response.content
    if len(data) > CONFIG["max_image_bytes"]:
        raise ValueError("Ảnh vượt giới hạn tải")
    image = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
    image.load()
    image = fit_image(image)
    with IMAGE_CACHE_LOCK:
        IMAGE_CACHE[cache_key] = image
        IMAGE_CACHE.move_to_end(cache_key)
        while len(IMAGE_CACHE) > CONFIG["image_cache_size"]:
            IMAGE_CACHE.popitem(last=False)
    return image


def prefetch_documents(documents):
    # Trả về (prepared, failed). prepared giữ nguyên thứ tự đầu vào.
    prepared, failed = [], []
    futures = {}
    for document in documents:
        url = document.get("image")
        if url:
            futures[document["id"]] = DOWNLOAD_POOL.submit(download_image, url)
    for document in documents:
        doc_id = document["id"]
        text = (document.get("text") or "").strip() or None
        image = None
        future = futures.get(doc_id)
        if future is not None:
            try:
                image = future.result()
            except Exception as exc:
                if not text:
                    failed.append({"id": doc_id, "error": f"{type(exc).__name__}: {exc}"[:300]})
                    continue
                failed.append({"id": doc_id, "error": f"image unavailable, dùng text: {exc}"[:300]})
        if image is None and not text:
            failed.append({"id": doc_id, "error": "document không có image lẫn text"})
            continue
        prepared.append({"id": doc_id, "image": image, "text": text})
    return prepared, failed


def _as_id_lists(value):
    if torch.is_tensor(value):
        return [row.tolist() for row in value]
    if isinstance(value, np.ndarray):
        return [list(map(int, row)) for row in value]
    if len(value) > 0 and not isinstance(value[0], (list, tuple, np.ndarray, torch.Tensor)):
        return [list(map(int, value))]
    return [list(map(int, row)) for row in value]


def _to_tensor(value):
    if torch.is_tensor(value):
        return value
    if isinstance(value, np.ndarray):
        return torch.from_numpy(value)
    if isinstance(value, (list, tuple)):
        try:
            return torch.as_tensor(np.asarray(value))
        except Exception:
            return value
    return value


def _kept_token_indices(row, max_length, special_ids):
    # Bản trả-về-index của Qwen3VLReranker.truncate_tokens_optimized, áp lên
    # row[:-5] rồi luôn giữ 5 token cuối. Cần index (không phải value) để cắt các
    # mảng song song theo token đúng từng vị trí.
    head = row[:-5]
    kept = list(range(len(head)))
    if len(head) > max_length:
        special = set(special_ids)
        budget = max_length - sum(1 for token in head if token in special)
        kept, taken = [], 0
        for index, token in enumerate(head):
            if token in special:
                kept.append(index)
            elif taken < budget:
                kept.append(index)
                taken += 1
    kept.extend(range(max(0, len(row) - 5), len(row)))
    return kept


def _row_lengths(value):
    # Độ dài từng hàng nếu value là batch chuỗi theo token; None nếu không phải.
    try:
        if torch.is_tensor(value) or isinstance(value, np.ndarray):
            return [int(value.shape[1])] * int(value.shape[0]) if value.ndim == 2 else None
        if (
            isinstance(value, (list, tuple))
            and value
            and isinstance(value[0], (list, tuple, np.ndarray, torch.Tensor))
        ):
            return [len(row) for row in value]
    except Exception:
        return None
    return None


def tokenize_batch(pairs):
    # Bản batch của Qwen3VLReranker.tokenize(). Giữ nguyên: chat template chính thức,
    # do_resize=False (process_vision_info đã smart-resize), truncation giữ special
    # token + 5 token cuối, rồi pad trái. Khác biệt duy nhất là ép return_tensors=None
    # để input_ids chắc chắn là list và batch có độ dài lệch nhau vẫn pad được.
    texts = PROCESSOR.apply_chat_template(pairs, tokenize=False, add_generation_prompt=True)
    if isinstance(texts, str):
        texts = [texts]
    images, videos, video_kwargs = process_vision_info(
        pairs,
        image_patch_size=CONFIG["image_patch_size"],
        return_video_kwargs=True,
        return_video_metadata=True,
    )
    if videos is not None:
        raise RuntimeError("Rerank path chỉ nhận image/text, nhưng nhận được video input.")
    processor_kwargs = dict(
        text=texts,
        images=images,
        videos=None,
        video_metadata=None,
        truncation=False,
        padding=False,
        do_resize=False,
        **video_kwargs,
    )
    try:
        # return_tensors=None ép input_ids ra list, nên batch có độ dài lệch nhau
        # không bị tokenizer từ chối tạo tensor trước khi kịp pad.
        inputs = PROCESSOR(return_tensors=None, **processor_kwargs)
    except (TypeError, ValueError):
        inputs = PROCESSOR(**processor_kwargs)
    inputs = dict(inputs)
    special_ids = TOKENIZER.all_special_ids
    raw_ids = _as_id_lists(inputs["input_ids"])
    original_lengths = [len(row) for row in raw_ids]

    id_lists, keep_indices = [], []
    for row in raw_ids:
        # input_ids đi qua đúng hàm official; keep_indices chỉ để cắt các mảng
        # song song theo token cho khớp từng vị trí, và được đối chiếu ngay dưới.
        truncated = RERANKER.truncate_tokens_optimized(
            row[:-5], CONFIG["max_length"], special_ids
        ) + row[-5:]
        kept = _kept_token_indices(row, CONFIG["max_length"], special_ids)
        if len(kept) != len(truncated):
            raise RuntimeError(
                "Truncation theo index lệch official truncate_tokens_optimized "
                f"({len(kept)} != {len(truncated)})."
            )
        id_lists.append(truncated)
        keep_indices.append(kept)

    padded = TOKENIZER.pad({"input_ids": id_lists}, padding=True, return_tensors="pt")
    padded_length = int(padded["input_ids"].shape[1])
    pad_left = getattr(TOKENIZER, "padding_side", "left") == "left"

    batch = {}
    for key, value in inputs.items():
        if key in ("input_ids", "attention_mask"):
            continue
        if _row_lengths(value) == original_lengths:
            # Mảng song song theo token — mm_token_type_ids là cái model bắt buộc.
            # get_rope_index index nó bằng attention_mask, nên nó PHẢI được cắt và
            # pad y hệt input_ids. Wrapper official bỏ qua bước này, và đó chính là
            # chỗ process() chết với "only integer tensors ... can be converted to
            # an index": mm_token_type_ids vẫn còn là list chưa cắt, chưa pad.
            aligned = []
            for row, kept in zip(_as_id_lists(value), keep_indices):
                kept_row = [row[index] for index in kept]
                padding = [0] * (padded_length - len(kept_row))  # 0 = text token
                aligned.append(padding + kept_row if pad_left else kept_row + padding)
            batch[key] = torch.as_tensor(aligned, dtype=torch.long)
        else:
            batch[key] = _to_tensor(value)
    batch["input_ids"] = padded["input_ids"]
    batch["attention_mask"] = padded["attention_mask"]
    # BatchFeature (không phải dict) để official process() gọi được .to(device).
    return BatchFeature(data=batch)


# Vá tokenize của chính instance official. format_mm_instruction, compute_scores
# và mọi tham số ngữ nghĩa vẫn nguyên bản; chỉ bước ghép tensor được sửa. Nhờ vậy
# RERANKER.process() chạy lại được, và parity gate mới thực sự so được
# "official từng cặp" với "batched".
RERANKER.tokenize = tokenize_batch


@torch.inference_mode()
def score_batch(batch):
    # Giống Qwen3VLReranker.compute_scores: last token của mỗi hàng (pad trái nên
    # luôn là token thật) chiếu qua w_yes - w_no rồi sigmoid.
    gpu_batch = {
        key: (value.to(DEVICE, non_blocking=True) if torch.is_tensor(value) else value)
        for key, value in batch.items()
    }
    hidden = MODEL(**gpu_batch).last_hidden_state[:, -1]
    scores = SCORE_LINEAR(hidden)
    return torch.sigmoid(scores).squeeze(-1).float().reshape(-1).cpu().tolist()


def build_pairs(instruction, query, prepared):
    return [
        RERANKER.format_mm_instruction(
            query, None, None,
            document.get("text"), document.get("image"), None,
            instruction=instruction,
        )
        for document in prepared
    ]


def score_prepared(instruction, query, prepared, batch_size=None):
    if not prepared:
        return [], {"tokenize_ms": 0.0, "inference_ms": 0.0, "batch_size": STATE["batch_size"]}
    pairs = build_pairs(instruction, query, prepared)
    batch_size = int(batch_size or STATE["batch_size"])
    scores, tokenize_ms, inference_ms = [], 0.0, 0.0
    index = 0
    while index < len(pairs):
        chunk = pairs[index:index + batch_size]
        batch = None
        try:
            t0 = time.perf_counter()
            batch = tokenize_batch(chunk)
            t1 = time.perf_counter()
            with MODEL_LOCK:
                chunk_scores = score_batch(batch)
            t2 = time.perf_counter()
        except torch.cuda.OutOfMemoryError:
            batch = None
            gc.collect()
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            STATE["batch_size"] = batch_size
            print(f"OOM → hạ batch_size xuống {batch_size}")
            continue
        tokenize_ms += (t1 - t0) * 1000
        inference_ms += (t2 - t1) * 1000
        scores.extend(chunk_scores)
        index += len(chunk)
    return scores, {
        "tokenize_ms": round(tokenize_ms, 1),
        "inference_ms": round(inference_ms, 1),
        "batch_size": batch_size,
    }


def cache_key_for(instruction, query, documents, top_k):
    payload = json.dumps({
        "model": CONFIG["model_id"],
        "revision": CONFIG["model_revision"],
        "max_pixels": RERANKER.max_pixels,
        "max_length": CONFIG["max_length"],
        "instruction": instruction,
        "query": query,
        "documents": [[d["id"], d.get("image"), d.get("text")] for d in documents],
        "top_k": top_k,
    }, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def rerank(query, documents, instruction=None, top_k=None, use_cache=True):
    started = time.perf_counter()
    instruction = (instruction or CONFIG["default_instruction"]).strip()
    query = (query or "").strip()
    if not query:
        raise ValueError("query rỗng")

    # Trùng id thì giữ bản đầu tiên; backend gửi submit_keyframe_id nên trùng là bug phía trên.
    unique, seen = [], set()
    for document in documents:
        doc_id = str(document.get("id") or "").strip()
        if not doc_id or doc_id in seen:
            continue
        seen.add(doc_id)
        unique.append({"id": doc_id, "image": document.get("image"), "text": document.get("text")})

    key = cache_key_for(instruction, query, unique, top_k) if use_cache else None
    if key is not None:
        with RESULT_CACHE_LOCK:
            hit = RESULT_CACHE.get(key)
            if hit is not None:
                RESULT_CACHE.move_to_end(key)
                cached = dict(hit)
                cached["cached"] = True
                return cached

    t_download = time.perf_counter()
    prepared, failed = prefetch_documents(unique)
    download_ms = (time.perf_counter() - t_download) * 1000

    scores, timing = score_prepared(instruction, query, prepared)
    ranked = sorted(
        ({"id": document["id"], "score": float(score)} for document, score in zip(prepared, scores)),
        key=lambda item: -item["score"],
    )
    if top_k:
        ranked = ranked[:int(top_k)]
    for rank, item in enumerate(ranked):
        item["rank"] = rank

    result = {
        "model": CONFIG["model_id"],
        "instruction": instruction,
        "query": query,
        "scores": ranked,
        "failed": failed,
        "requested": len(unique),
        "scored": len(scores),
        "returned": len(ranked),
        "image_token_budget": RERANKER.max_pixels // (IMAGE_FACTOR * IMAGE_FACTOR),
        "download_ms": round(download_ms, 1),
        "tokenize_ms": timing["tokenize_ms"],
        "inference_ms": timing["inference_ms"],
        "batch_size": timing["batch_size"],
        "latency_ms": round((time.perf_counter() - started) * 1000, 1),
        "cached": False,
    }
    if key is not None:
        with RESULT_CACHE_LOCK:
            RESULT_CACHE[key] = result
            while len(RESULT_CACHE) > CONFIG["result_cache_size"]:
                RESULT_CACHE.popitem(last=False)
    return result


def synthetic_image(seed, width=1280, height=720):
    rng = np.random.default_rng(seed)
    gradient = np.linspace(0, 255, width, dtype=np.uint8)
    array = np.repeat(gradient[None, :, None], height, axis=0).repeat(3, axis=2)
    array = (array * rng.uniform(0.4, 1.0, size=(1, 1, 3))).astype(np.uint8)
    image = Image.fromarray(array)
    draw = ImageDraw.Draw(image)
    for _ in range(6):
        x0, y0 = rng.integers(0, width - 200), rng.integers(0, height - 150)
        draw.rectangle(
            [int(x0), int(y0), int(x0) + int(rng.integers(60, 200)), int(y0) + int(rng.integers(50, 150))],
            fill=tuple(int(value) for value in rng.integers(0, 255, size=3)),
        )
    return fit_image(image)


def sample_documents(count):
    # Ưu tiên URL keyframe thật trong CONFIG; thiếu thì bù bằng ảnh synthetic.
    documents = []
    urls = list(CONFIG["sample_image_urls"])[:count]
    for i, url in enumerate(urls):
        documents.append({"id": f"URL{i:03d}", "image": url})
    for i in range(count - len(urls)):
        documents.append({"id": f"SYN{i:03d}", "image": None, "_pil": synthetic_image(i)})
    return documents


def prepare_sample(count):
    # Bỏ qua network cho ảnh synthetic: nạp thẳng PIL vào prepared.
    documents = sample_documents(count)
    with_url = [d for d in documents if d.get("image")]
    prepared, failed = prefetch_documents(with_url) if with_url else ([], [])
    for document in documents:
        if document.get("_pil") is not None:
            prepared.append({"id": document["id"], "image": document["_pil"], "text": None})
    if failed:
        print("WARNING: sample image tải lỗi:", failed[:3])
    return prepared


if CONFIG["warmup_on_start"]:
    warm_prepared = prepare_sample(2)
    warm_scores, warm_timing = score_prepared(
        CONFIG["default_instruction"], CONFIG["sample_query"], warm_prepared, batch_size=2
    )
    print("warmup scores:", [round(score, 4) for score in warm_scores], warm_timing)
    print("vram sau warmup (GiB):", round(torch.cuda.max_memory_allocated() / 2**30, 2))


## 5. Parity gate: batched scores vs `process()` chính thức

Batch làm thay đổi shape và padding, còn forward thì BF16 — nên hai đường phải được
so trực tiếp. Gate này chấm cùng bộ document bằng `RERANKER.process()` (vòng lặp tuần tự
official, **một cặp một forward**) và bằng engine batch, rồi kiểm tra sai số tuyệt đối.

Lưu ý phạm vi: cả hai đường đều đi qua `tokenize` đã vá ở cell trước — bản official nguyên
gốc không chạy được với transformers hiện tại. Thứ gate này chứng minh là **batching không
làm đổi điểm**, còn `format_mm_instruction` / `compute_scores` / pixel budget thì vẫn là
official nguyên bản ở cả hai phía.

Nếu gate fail: hạ `batch_size` về 1 (engine tự động khớp official) và mở issue trước
khi đem ra thi.


In [ ]:
if CONFIG["run_parity_gate"]:
    parity_prepared = prepare_sample(CONFIG["parity_samples"])
    parity_documents = [
        {key: value for key, value in document.items() if key != "id" and value is not None}
        for document in parity_prepared
    ]

    t0 = time.perf_counter()
    with MODEL_LOCK:
        official_scores = RERANKER.process({
            "instruction": CONFIG["default_instruction"],
            "query": {"text": CONFIG["sample_query"]},
            "documents": parity_documents,
        })
    official_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    batched_scores, batched_timing = score_prepared(
        CONFIG["default_instruction"], CONFIG["sample_query"], parity_prepared
    )
    batched_ms = (time.perf_counter() - t0) * 1000

    diffs = [abs(a - b) for a, b in zip(official_scores, batched_scores)]
    max_diff = max(diffs) if diffs else 0.0
    official_order = [i for i, _ in sorted(enumerate(official_scores), key=lambda kv: -kv[1])]
    batched_order = [i for i, _ in sorted(enumerate(batched_scores), key=lambda kv: -kv[1])]

    print(json.dumps({
        "documents": len(parity_prepared),
        "official_scores": [round(score, 5) for score in official_scores],
        "batched_scores": [round(score, 5) for score in batched_scores],
        "max_abs_diff": round(max_diff, 6),
        "mean_abs_diff": round(sum(diffs) / len(diffs), 6) if diffs else 0.0,
        "official_ms": round(official_ms, 1),
        "batched_ms": round(batched_ms, 1),
        "speedup": round(official_ms / batched_ms, 2) if batched_ms else None,
        "batch_size": batched_timing["batch_size"],
        "order_identical": official_order == batched_order,
    }, indent=2))

    if max_diff > CONFIG["parity_max_abs_diff"]:
        raise RuntimeError(
            f"Parity gate fail: max_abs_diff={max_diff:.6f} > {CONFIG['parity_max_abs_diff']}. "
            "Đặt CONFIG['batch_size']=1 rồi chạy lại."
        )
    if official_order != batched_order:
        print("WARNING: thứ tự lệch ở các cặp gần bằng điểm — kiểm tra lại nếu gap > 0.05.")
    print("Parity gate PASS.")
else:
    print("Bỏ qua parity gate (CONFIG['run_parity_gate']=False).")


## 6. Autotune batch size và benchmark latency (tuỳ chọn)

`reranker.md` §11 yêu cầu benchmark thật trước khi chốt 2B hay 8B. Cell này đo
throughput theo `batch_size` và theo `image_token_budget` để chọn điểm vận hành:
với AIC interactive retrieval, ngân sách thực tế là **rerank 150–300 candidate trong
vài giây**, không phải "càng nét càng tốt".


In [ ]:
def measure(prepared, batch_size):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    score_prepared(CONFIG["default_instruction"], CONFIG["sample_query"], prepared, batch_size=batch_size)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return {
        "batch_size": batch_size,
        "docs": len(prepared),
        "seconds": round(elapsed, 3),
        "docs_per_s": round(len(prepared) / elapsed, 2),
        "peak_vram_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    }


if CONFIG["autotune_batch_size"]:
    tune_prepared = prepare_sample(max(CONFIG["batch_candidates"]) * 2)
    vram_budget = GPU_VRAM_GIB * CONFIG["max_vram_fraction"]
    rows, best = [], None
    for candidate in CONFIG["batch_candidates"]:
        try:
            row = measure(tune_prepared, candidate)
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            print(f"batch_size={candidate}: OOM, dừng autotune.")
            break
        rows.append(row)
        print(row)
        if row["peak_vram_gib"] > vram_budget:
            print(f"batch_size={candidate} vượt ngân sách VRAM {vram_budget:.1f} GiB; dừng.")
            break
        if best is None or row["docs_per_s"] > best["docs_per_s"]:
            best = row
    if best:
        STATE["batch_size"] = best["batch_size"]
        print("Chọn batch_size =", STATE["batch_size"], "|", best)
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Giữ batch_size =", STATE["batch_size"])


if CONFIG["run_latency_benchmark"]:
    baseline_max_pixels = RERANKER.max_pixels
    benchmark_rows = []
    for budget in CONFIG["benchmark_token_budgets"]:
        RERANKER.max_pixels = budget * IMAGE_FACTOR * IMAGE_FACTOR
        with IMAGE_CACHE_LOCK:
            IMAGE_CACHE.clear()  # cache key gắn với max_pixels; xoá cho sạch RAM
        bench_prepared = prepare_sample(CONFIG["benchmark_documents"])
        try:
            row = measure(bench_prepared, STATE["batch_size"])
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            print(f"image_token_budget={budget}: OOM")
            continue
        row["image_token_budget"] = budget
        row["ms_per_150_docs"] = round(150 / row["docs_per_s"] * 1000, 1)
        benchmark_rows.append(row)
        print(row)
    RERANKER.max_pixels = baseline_max_pixels
    with IMAGE_CACHE_LOCK:
        IMAGE_CACHE.clear()
    gc.collect()
    torch.cuda.empty_cache()
    print("\nBảng benchmark:")
    print(json.dumps(benchmark_rows, indent=2))
    print(
        "\nChọn image_token_budget lớn nhất mà ms_per_150_docs vẫn nằm trong "
        "ngân sách latency của console, rồi đặt lại CONFIG['image_token_budget'] và Run all."
    )
else:
    print("Bỏ qua latency benchmark (CONFIG['run_latency_benchmark']=False).")


## 7. FastAPI — contract backend tiêu thụ

```http
POST /rerank
Authorization: Bearer <AIC26_QWEN_RERANKER_TOKEN>
```

```json
{
  "query": "a man riding a bicycle on a street",
  "instruction": "optional — mặc định là instruction keyframe của server",
  "top_k": 100,
  "documents": [
    {"id": "L21/L21_V001/042", "image": "https://.../042.jpg"},
    {"id": "L21/L21_V003/013", "image": "https://.../013.jpg", "text": "OCR: giá xăng tăng"}
  ]
}
```

```json
{
  "scores": [{"id": "L21/L21_V003/013", "score": 0.982, "rank": 0}],
  "failed": [],
  "scored": 200, "returned": 100,
  "latency_ms": 2140, "download_ms": 310, "inference_ms": 1760
}
```

Backend chỉ cần reorder theo `scores` rồi **append các id không có trong `scores`**
(ảnh 404, hoặc bị `top_k` cắt) theo đúng thứ tự PE cũ. `id` nên là `submit_keyframe_id`
để không phải map ngược.


In [ ]:
from typing import Optional

from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field


class RerankDocument(BaseModel):
    id: str = Field(min_length=1, max_length=200)
    image: Optional[str] = Field(default=None, max_length=2000)
    text: Optional[str] = Field(default=None, max_length=4000)


class RerankRequest(BaseModel):
    query: str = Field(min_length=1, max_length=2000)
    documents: list[RerankDocument] = Field(min_length=1, max_length=CONFIG["max_documents"])
    instruction: Optional[str] = Field(default=None, max_length=1000)
    top_k: Optional[int] = Field(default=None, ge=1, le=CONFIG["max_documents"])
    use_cache: bool = True


def require_auth(authorization: Optional[str]):
    expected = f"Bearer {SERVICE_TOKEN}"
    if not authorization or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="Unauthorized")


app = FastAPI(title="AIC26 Qwen3-VL-Reranker-8B Worker", version="1.0.0")


@app.get("/health")
def health():
    return {
        "ok": True,
        "model": CONFIG["model_id"],
        "revision": CONFIG["model_revision"],
        "gpu": GPU_NAME,
        "dtype": CONFIG["model_weight_dtype"],
        "attn_implementation": ATTENTION_IMPLEMENTATION,
        "default_instruction": CONFIG["default_instruction"],
        "image_token_budget": RERANKER.max_pixels // (IMAGE_FACTOR * IMAGE_FACTOR),
        "max_documents": CONFIG["max_documents"],
        "max_length": CONFIG["max_length"],
        "batch_size": STATE["batch_size"],
        "image_cache": len(IMAGE_CACHE),
        "vram_reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 2),
    }


@app.post("/rerank")
def rerank_endpoint(req: RerankRequest, authorization: Optional[str] = Header(None)):
    require_auth(authorization)
    try:
        return rerank(
            req.query,
            [document.model_dump() for document in req.documents],
            instruction=req.instruction,
            top_k=req.top_k,
            use_cache=req.use_cache,
        )
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    except torch.cuda.OutOfMemoryError as exc:
        gc.collect()
        torch.cuda.empty_cache()
        raise HTTPException(status_code=503, detail=f"GPU OOM: {exc}") from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Rerank failed: {exc}") from exc


@app.post("/cache/clear")
def clear_cache(authorization: Optional[str] = Header(None)):
    require_auth(authorization)
    with IMAGE_CACHE_LOCK:
        images = len(IMAGE_CACHE)
        IMAGE_CACHE.clear()
    with RESULT_CACHE_LOCK:
        results = len(RESULT_CACHE)
        RESULT_CACHE.clear()
    gc.collect()
    torch.cuda.empty_cache()
    return {"ok": True, "cleared_images": images, "cleared_results": results}


print("Routes ready:", [route.path for route in app.routes])


## 8. Chạy server


In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

previous_server = globals().get("api_server")
previous_thread = globals().get("api_thread")
if previous_server is not None and previous_thread is not None and previous_thread.is_alive():
    previous_server.should_exit = True
    previous_thread.join(timeout=15)
elif previous_thread is not None and previous_thread.is_alive():
    raise RuntimeError("Một API thread cũ vẫn chạy. Restart runtime rồi Run all.")

api_server = uvicorn.Server(
    uvicorn.Config(app, host="0.0.0.0", port=CONFIG["port"], log_level="info")
)


def run_api():
    api_server.run()


api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(2)
print(json.dumps(
    requests.get(f"http://127.0.0.1:{CONFIG['port']}/health", timeout=15).json(),
    indent=2, ensure_ascii=False,
))


## 9. Smoke test cục bộ

Đặt `SMOKE_IMAGE_URLS` bằng vài URL keyframe thật (`MediaUrlBuilder.keyframe_url`) và một
câu query khớp với một trong số đó. Nếu ảnh đúng không leo lên top thì vấn đề nằm ở
instruction/`image_token_budget`, không phải ở integration.


In [ ]:
RUN_SMOKE_TEST = False
SMOKE_QUERY = "a news anchor sitting at a desk in a television studio"
SMOKE_IMAGE_URLS = [
    # "https://<media-base>/Keyframes/Keyframes_L21/L21_V001/042.jpg",
]

if RUN_SMOKE_TEST:
    if not SMOKE_IMAGE_URLS:
        raise RuntimeError("Điền SMOKE_IMAGE_URLS trước khi bật RUN_SMOKE_TEST.")
    payload = {
        "query": SMOKE_QUERY,
        "top_k": 10,
        "documents": [
            {"id": url.rsplit("/Keyframes/", 1)[-1], "image": url}
            for url in SMOKE_IMAGE_URLS
        ],
    }
    response = requests.post(
        f"http://127.0.0.1:{CONFIG['port']}/rerank",
        json=payload,
        headers={"Authorization": f"Bearer {SERVICE_TOKEN}"},
        timeout=300,
    )
    print(response.status_code)
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))
else:
    print("Smoke test tắt (RUN_SMOKE_TEST=False).")


## 10. Cloudflare quick tunnel → `PUBLIC_URL`

Dùng đường dẫn tuyệt đối của binary đã tải nên không phụ thuộc PATH của Colab.
URL quick tunnel đổi mỗi lần restart tunnel, và tab Colab phải giữ kết nối. Muốn ổn định
lâu dài thì thay bằng named tunnel hoặc GPU VM.


In [ ]:
import queue
import subprocess

if globals().get("tunnel_proc") and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    tunnel_proc.wait(timeout=10)

tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{CONFIG['port']}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
log_queue = queue.Queue()


def read_tunnel_log():
    for line in tunnel_proc.stdout:
        log_queue.put(line)


threading.Thread(target=read_tunnel_log, daemon=True).start()

PUBLIC_URL = None
deadline = time.time() + 60
while time.time() < deadline and PUBLIC_URL is None:
    try:
        line = log_queue.get(timeout=1)
    except queue.Empty:
        continue
    print(line, end="")
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)

assert PUBLIC_URL, "Không tìm thấy tunnel URL. Chạy lại cell này và đọc log Cloudflare ở trên."
print("\nPUBLIC_URL =", PUBLIC_URL)
print("Backend .env: QWEN_RERANKER_URL=" + PUBLIC_URL)
print(f"Backend .env: QWEN_RERANKER_TOKEN=<giá trị secret {CONFIG['service_token_secret']}>")
print("Remote health:", json.dumps(
    requests.get(PUBLIC_URL + "/health", timeout=30).json(), ensure_ascii=False,
))


## 11. Nối vào backend AIC26

### `backend/.env`

```dotenv
QWEN_RERANKER_ENABLED=true
QWEN_RERANKER_URL=https://....trycloudflare.com
QWEN_RERANKER_TOKEN=<secret AIC26_QWEN_RERANKER_TOKEN>
QWEN_RERANKER_MODEL=8b
QWEN_RERANKER_CANDIDATES=200      # candidate_k gửi lên rerank
QWEN_RERANKER_TIMEOUT=25
```

### Điểm chèn

Trong `SearchService._run_image_pe()`, ngay **sau** bước union query-variant
(`fused = sorted(best.values(), ...)`) và **trước** khi dựng `ChannelHit`:

```python
fused = sorted(best.values(), key=lambda r: -r["score"])[:candidate_k]
fused = await self._rerank_visual(cfg, fused)      # fail-open, xem dưới
fused = fused[:top_k]
```

`simple_image_search()` là code path thứ hai (`Pure PE→Milvus`) — nếu không hook luôn ở
đó thì tab Simple Search vẫn chạy PE thuần.

### Fail-open là bắt buộc

`retrieve()` đang bọc mỗi channel bằng `except Exception: hits = []`. Nếu để exception của
reranker thoát ra khỏi `_run_image_pe()`, **toàn bộ** kết quả PE biến mất chỉ vì Colab
tunnel rớt. Reranker là refinement tuỳ chọn, không được thành single point of failure:

```python
async def _rerank_visual(self, cfg, candidates):
    if not self.reranker.enabled or not candidates:
        return candidates
    try:
        ranked = await self.reranker.rerank(
            query=(cfg.get("queries_en") or [""])[0],   # query chính, KHÔNG rerank từng variant
            documents=[
                {"id": r["submit_keyframe_id"],
                 "image": self.media.keyframe_url(r["video_id"], int(r["keyframe_n"]))}
                for r in candidates
            ],
        )
    except Exception:
        return candidates                               # giữ nguyên thứ tự PE
    by_id = {r["submit_keyframe_id"]: r for r in candidates}
    ordered = [by_id.pop(item["id"]) for item in ranked if item["id"] in by_id]
    return ordered + [by_id[key] for key in by_id]      # ảnh lỗi/bị cắt về cuối, đúng thứ tự cũ
```

### Vì sao chèn ở đây

- Qwen chấm `(query, image)` nên phải làm việc ở **frame level**, trước `group_by_video`.
- RRF hiện tại tính điểm bằng `weight / (k + rank + 1)` chứ không cộng cosine, nên chỉ cần
  **rank mới** — không phải calibrate `PE cosine 0.37` với `Qwen score 0.91`.
- Rerank sau global RRF sẽ để điểm ảnh thấp của Qwen phá tín hiệu OCR/speech/audio trên
  những frame mà bằng chứng nằm ở âm thanh chứ không ở hình.

### Ba con số cần đo trước khi chốt

`PE only` · `PE + Reranker top-150` · `PE + Reranker top-300`, so R@1/R@5/R@20 và latency
p50/p95. Reranker chỉ sửa được ranking trong tập candidate — nó **không** cứu được frame
mà PE để rơi ngoài `candidate_k` (giới hạn recall đó là việc của Qwen3-VL-Embedding).

### Khi có thêm Qwen3-VL-Embedding index

Service này dùng lại nguyên vẹn: union candidate của PE và Qwen embedding rồi gửi cùng
một `POST /rerank`; không phải sửa gì trong notebook.
